# Problem Statement

## **Business Context**

"Visit with Us," a leading travel company, is revolutionizing the tourism industry by leveraging data-driven strategies to optimize operations and customer engagement. While introducing a new package offering, such as the Wellness Tourism Package, the company faces challenges in targeting the right customers efficiently. The manual approach to identifying potential customers is inconsistent, time-consuming, and prone to errors, leading to missed opportunities and suboptimal campaign performance.

To address these issues, the company aims to implement a scalable and automated system that integrates customer data, predicts potential buyers, and enhances decision-making for marketing strategies. By utilizing an MLOps pipeline, the company seeks to achieve seamless integration of data preprocessing, model development, deployment, and CI/CD practices for continuous improvement. This system will ensure efficient targeting of customers, timely updates to the predictive model, and adaptation to evolving customer behaviors, ultimately driving growth and customer satisfaction.


## **Objective**

As an MLOps Engineer at "Visit with Us," your responsibility is to design and deploy an MLOps pipeline on GitHub to automate the end-to-end workflow for predicting customer purchases. The primary objective is to build a model that predicts whether a customer will purchase the newly introduced Wellness Tourism Package before contacting them. The pipeline will include data cleaning, preprocessing, transformation, model building, training, evaluation, and deployment, ensuring consistent performance and scalability. By leveraging GitHub Actions for CI/CD integration, the system will enable automated updates, streamline model deployment, and improve operational efficiency. This robust predictive solution will empower policymakers to make data-driven decisions, enhance marketing strategies, and effectively target potential customers, thereby driving customer acquisition and business growth.

## **Data Description**

The dataset contains customer and interaction data that serve as key attributes for predicting the likelihood of purchasing the Wellness Tourism Package. The detailed attributes are:

**Customer Details**
- **CustomerID:** Unique identifier for each customer.
- **ProdTaken:** Target variable indicating whether the customer has purchased a package (0: No, 1: Yes).
- **Age:** Age of the customer.
- **TypeofContact:** The method by which the customer was contacted (Company Invited or Self Inquiry).
- **CityTier:** The city category based on development, population, and living standards (Tier 1 > Tier 2 > Tier 3).
- **Occupation:** Customer's occupation (e.g., Salaried, Freelancer).
- **Gender:** Gender of the customer (Male, Female).
- **NumberOfPersonVisiting:** Total number of people accompanying the customer on the trip.
- **PreferredPropertyStar:** Preferred hotel rating by the customer.
- **MaritalStatus:** Marital status of the customer (Single, Married, Divorced).
- **NumberOfTrips:** Average number of trips the customer takes annually.
- **Passport:** Whether the customer holds a valid passport (0: No, 1: Yes).
- **OwnCar:** Whether the customer owns a car (0: No, 1: Yes).
- **NumberOfChildrenVisiting:** Number of children below age 5 accompanying the customer.
- **Designation:** Customer's designation in their current organization.
- **MonthlyIncome:** Gross monthly income of the customer.

**Customer Interaction Data**
- **PitchSatisfactionScore:** Score indicating the customer's satisfaction with the sales pitch.
- **ProductPitched:** The type of product pitched to the customer.
- **NumberOfFollowups:** Total number of follow-ups by the salesperson after the sales pitch.-
- **DurationOfPitch:** Duration of the sales pitch delivered to the customer.


## Local Environment Setup

For local Jupyter execution, store secrets in a `.env` file in the project root. The generated scripts load this file automatically, while GitHub Actions will continue to use repository secrets.

Update `.env` with your real Hugging Face token before running the data registration, preparation, training, or hosting scripts:

```text
HF_TOKEN=hf_your_token_here
NGROK_AUTH_TOKEN=your_ngrok_token_here
```


In [8]:
from pathlib import Path
import os
from dotenv import load_dotenv

env_path = Path(".env")
if not env_path.exists():
    env_path.write_text("""HF_TOKEN=
NGROK_AUTH_TOKEN=
""")

load_dotenv(env_path, override=True)
print("HF_TOKEN loaded:", bool(os.getenv("HF_TOKEN")))
print("NGROK_AUTH_TOKEN loaded:", bool(os.getenv("NGROK_AUTH_TOKEN")))


HF_TOKEN loaded: True
NGROK_AUTH_TOKEN loaded: True


# Model Building

In [2]:
# Create a master folder to keep all files created when executing the below code cells
import os
os.makedirs("tourism_project", exist_ok=True)

In [3]:
# Create a folder for storing the model building files
os.makedirs("tourism_project/model_building", exist_ok=True)

## Data Registration

In [4]:
import shutil

os.makedirs("tourism_project/data", exist_ok=True)

# Keep a copy inside the project folder for local notebook runs and GitHub Actions.
if os.path.exists("tourism.csv"):
    shutil.copy("tourism.csv", "tourism_project/data/tourism.csv")


Once the **data** folder created after executing the above cell, please upload the **tourism.csv** in to the folder

### Prerequisites for Data Registration

Before running the data registration script:

1. Create a Hugging Face account and generate an access token with write permission.
2. For local Jupyter runs, set the token as `HF_TOKEN` in the project `.env` file. For GitHub Actions, add `HF_TOKEN` to repository secrets.
3. Replace `<---your-huggingface-username--->` in the scripts with your Hugging Face username.
4. Ensure `tourism.csv` is available inside `tourism_project/data/`.


In [5]:
%%writefile tourism_project/model_building/data_register.py
from huggingface_hub.utils import RepositoryNotFoundError
from huggingface_hub import HfApi, create_repo
import os
from dotenv import load_dotenv

HF_USERNAME = "karthikei94"
DATASET_REPO_NAME = "tourism-customer-purchase-prediction"
repo_id = f"{HF_USERNAME}/{DATASET_REPO_NAME}"
repo_type = "dataset"

load_dotenv()

hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    raise ValueError("HF_TOKEN is missing. Add it to your .env file for local runs or GitHub secrets for Actions.")

api = HfApi(token=hf_token)

try:
    api.repo_info(repo_id=repo_id, repo_type=repo_type)
    print(f"Dataset repo '{repo_id}' already exists. Using it.")
except RepositoryNotFoundError:
    print(f"Dataset repo '{repo_id}' not found. Creating it...")
    create_repo(repo_id=repo_id, repo_type=repo_type, private=False, token=hf_token)
    print(f"Dataset repo '{repo_id}' created.")

api.upload_folder(
    folder_path="tourism_project/data",
    repo_id=repo_id,
    repo_type=repo_type,
)
print("Dataset folder uploaded successfully.")


Overwriting tourism_project/model_building/data_register.py


## Data Preparation

In [6]:
%%writefile tourism_project/model_building/prep.py
import os
from dotenv import load_dotenv
import pandas as pd
from sklearn.model_selection import train_test_split
from huggingface_hub import HfApi

HF_USERNAME = "karthikei94"
DATASET_REPO_NAME = "tourism-customer-purchase-prediction"
repo_id = f"{HF_USERNAME}/{DATASET_REPO_NAME}"

hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    raise ValueError("HF_TOKEN is missing. Add it to your .env file for local runs or GitHub secrets for Actions.")

api = HfApi(token=hf_token)
DATASET_PATH = f"hf://datasets/{repo_id}/tourism.csv"

df = pd.read_csv(DATASET_PATH)
print("Dataset loaded successfully.")

# Drop index/identifier fields that should not influence customer purchase prediction.
df = df.drop(columns=["Unnamed: 0", "CustomerID"], errors="ignore")

# Normalize inconsistent category spelling found in the dataset.
df["Gender"] = df["Gender"].replace({"Fe Male": "Female"})

target_col = "ProdTaken"
X = df.drop(columns=[target_col])
y = df[target_col]

Xtrain, Xtest, ytrain, ytest = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

Xtrain.to_csv("Xtrain.csv", index=False)
Xtest.to_csv("Xtest.csv", index=False)
ytrain.to_csv("ytrain.csv", index=False)
ytest.to_csv("ytest.csv", index=False)

for file_path in ["Xtrain.csv", "Xtest.csv", "ytrain.csv", "ytest.csv"]:
    api.upload_file(
        path_or_fileobj=file_path,
        path_in_repo=file_path,
        repo_id=repo_id,
        repo_type="dataset",
    )

print("Train/test split files uploaded successfully.")


Overwriting tourism_project/model_building/prep.py


## Model Training and Registration with Experimentation Tracking

### Experimentation and Tracking (Development Environment)

The cells below are used to validate model experimentation locally before converting the logic into the production training script. They start MLflow tracking, run hyperparameter tuning, log nested runs, and print model evaluation metrics.


In [7]:
!pip install mlflow==3.0.1 pyngrok==7.2.12 python-dotenv==1.1.1 -q


To expose the MLflow UI from a notebook environment, create an ngrok token from https://dashboard.ngrok.com/authtokens and paste it in the placeholder below. If you run MLflow only on your local machine, you can skip ngrok and open `http://127.0.0.1:5000` directly.


In [9]:
from pyngrok import ngrok
from dotenv import load_dotenv
import os
import subprocess

load_dotenv(".env", override=True)
ngrok_token = os.getenv("NGROK_AUTH_TOKEN")
if not ngrok_token:
    raise ValueError("NGROK_AUTH_TOKEN is missing. Add it to your .env file, or skip this cell if you only need local MLflow.")

process = subprocess.Popen(["mlflow", "ui", "--host", "127.0.0.1", "--port", "5000"])
public_url = ngrok.connect(5000, authtoken=ngrok_token).public_url
print("MLflow UI is available at:", public_url)


t=2026-06-27T08:20:02+0530 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:  authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.
ERROR:  
ERROR:  Sign up for an account: https://dashboard.ngrok.com/signup
ERROR:  Get your credential: https://dashboard.ngrok.com/get-started/your-authtoken
ERROR:  
ERROR:  ERR_NGROK_4018
ERROR:  https://ngrok.com/docs/errors/err_ngrok_4018
ERROR:  
t=2026-06-27T08:20:02+0530 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a sess

PyngrokNgrokError: The ngrok process errored on start: authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [10]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("tourism_package_prediction_dev")


2026/06/27 08:22:28 INFO mlflow.tracking.fluent: Experiment with name 'tourism_package_prediction_dev' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/568390953453601594', creation_time=1782528748477, experiment_id='568390953453601594', last_update_time=1782528748477, lifecycle_stage='active', name='tourism_package_prediction_dev', tags={}>

In [11]:
import joblib
import mlflow
import pandas as pd
import xgboost as xgb
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Load the dataset locally for development experimentation.
df = pd.read_csv("tourism.csv")
print("Dataset loaded successfully.")

df = df.drop(columns=["Unnamed: 0", "CustomerID"], errors="ignore")
df["Gender"] = df["Gender"].replace({"Fe Male": "Female"})

target_col = "ProdTaken"
X = df.drop(columns=[target_col])
y = df[target_col]

Xtrain, Xtest, ytrain, ytest = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features = [
    "Age", "CityTier", "DurationOfPitch", "NumberOfPersonVisiting",
    "NumberOfFollowups", "PreferredPropertyStar", "NumberOfTrips",
    "Passport", "PitchSatisfactionScore", "OwnCar",
    "NumberOfChildrenVisiting", "MonthlyIncome",
]

categorical_features = [
    "TypeofContact", "Occupation", "Gender", "ProductPitched",
    "MaritalStatus", "Designation",
]

numeric_transformer = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
categorical_transformer = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore"),
)

preprocessor = make_column_transformer(
    (numeric_transformer, numeric_features),
    (categorical_transformer, categorical_features),
)

class_weight = ytrain.value_counts()[0] / ytrain.value_counts()[1]
print(f"Class weight: {class_weight:.2f}")

xgb_model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=class_weight,
    random_state=42,
    n_jobs=-1,
)

model_pipeline = make_pipeline(preprocessor, xgb_model)

param_grid = {
    "xgbclassifier__n_estimators": [100, 150],
    "xgbclassifier__max_depth": [3, 5],
    "xgbclassifier__learning_rate": [0.05, 0.1],
    "xgbclassifier__subsample": [0.8, 1.0],
    "xgbclassifier__colsample_bytree": [0.8, 1.0],
    "xgbclassifier__reg_lambda": [1, 5],
}

with mlflow.start_run():
    grid_search = GridSearchCV(
        model_pipeline,
        param_grid,
        cv=3,
        scoring="f1",
        n_jobs=-1,
        verbose=1,
    )
    grid_search.fit(Xtrain, ytrain)

    results = grid_search.cv_results_
    for i, params in enumerate(results["params"]):
        with mlflow.start_run(nested=True):
            mlflow.log_params(params)
            mlflow.log_metric("cv_mean_f1_score", results["mean_test_score"][i])
            mlflow.log_metric("cv_std_f1_score", results["std_test_score"][i])

    best_model = grid_search.best_estimator_
    mlflow.log_params(grid_search.best_params_)
    mlflow.log_metric("best_cv_f1_score", grid_search.best_score_)

    threshold = 0.45
    train_proba = best_model.predict_proba(Xtrain)[:, 1]
    test_proba = best_model.predict_proba(Xtest)[:, 1]
    y_pred_train = (train_proba >= threshold).astype(int)
    y_pred_test = (test_proba >= threshold).astype(int)

    train_report = classification_report(ytrain, y_pred_train, output_dict=True)
    test_report = classification_report(ytest, y_pred_test, output_dict=True)
    tn, fp, fn, tp = confusion_matrix(ytest, y_pred_test).ravel()

    mlflow.log_metrics({
        "train_accuracy": train_report["accuracy"],
        "train_precision": train_report["1"]["precision"],
        "train_recall": train_report["1"]["recall"],
        "train_f1_score": train_report["1"]["f1-score"],
        "test_accuracy": test_report["accuracy"],
        "test_precision": test_report["1"]["precision"],
        "test_recall": test_report["1"]["recall"],
        "test_f1_score": test_report["1"]["f1-score"],
        "test_roc_auc": roc_auc_score(ytest, test_proba),
        "test_true_negative": tn,
        "test_false_positive": fp,
        "test_false_negative": fn,
        "test_true_positive": tp,
    })

    joblib.dump(best_model, "best_tourism_purchase_model_dev.joblib")
    mlflow.log_artifact("best_tourism_purchase_model_dev.joblib", artifact_path="model")

print("\n=== Classification Report - Training Set ===")
print(classification_report(ytrain, y_pred_train))
print("\n=== Classification Report - Test Set ===")
print(classification_report(ytest, y_pred_test))


Dataset loaded successfully.
Class weight: 4.18
Fitting 3 folds for each of 64 candidates, totalling 192 fits
🏃 View run rumbling-sponge-159 at: http://127.0.0.1:5000/#/experiments/568390953453601594/runs/09af929253da45288f52c40ad5610d26
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/568390953453601594
🏃 View run melodic-seal-765 at: http://127.0.0.1:5000/#/experiments/568390953453601594/runs/92d32d7726214536bf3765a3374d4cfd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/568390953453601594
🏃 View run redolent-whale-756 at: http://127.0.0.1:5000/#/experiments/568390953453601594/runs/433e2f031ba94449b743f7fe984b9a89
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/568390953453601594
🏃 View run traveling-cow-516 at: http://127.0.0.1:5000/#/experiments/568390953453601594/runs/507daef5b3cf4473840ab823eac22f42
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/568390953453601594
🏃 View run suave-croc-109 at: http://127.0.0.1:5000/#/experiments/5683909534

### Experimentation and Tracking (Production Environment)

The development code above is converted into the `train.py` script below so it can run automatically in GitHub Actions and register the trained model on Hugging Face Hub.


In [12]:
%%writefile tourism_project/model_building/train.py
import os
from dotenv import load_dotenv
import joblib
import mlflow
import pandas as pd
import xgboost as xgb
from huggingface_hub import HfApi, create_repo
from huggingface_hub.utils import RepositoryNotFoundError
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

HF_USERNAME = "karthikei94"
DATASET_REPO_NAME = "tourism-customer-purchase-prediction"
MODEL_REPO_NAME = "tourism-purchase-model"
DATASET_REPO_ID = f"{HF_USERNAME}/{DATASET_REPO_NAME}"
MODEL_REPO_ID = f"{HF_USERNAME}/{MODEL_REPO_NAME}"
MODEL_FILENAME = "best_tourism_purchase_model_v1.joblib"

load_dotenv()

mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000"))
mlflow.set_experiment("tourism-purchase-prediction")

hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    raise ValueError("HF_TOKEN is missing. Add it to your .env file for local runs or GitHub secrets for Actions.")

api = HfApi(token=hf_token)

Xtrain = pd.read_csv(f"hf://datasets/{DATASET_REPO_ID}/Xtrain.csv")
Xtest = pd.read_csv(f"hf://datasets/{DATASET_REPO_ID}/Xtest.csv")
ytrain = pd.read_csv(f"hf://datasets/{DATASET_REPO_ID}/ytrain.csv").squeeze("columns")
ytest = pd.read_csv(f"hf://datasets/{DATASET_REPO_ID}/ytest.csv").squeeze("columns")

numeric_features = [
    "Age",
    "CityTier",
    "DurationOfPitch",
    "NumberOfPersonVisiting",
    "NumberOfFollowups",
    "PreferredPropertyStar",
    "NumberOfTrips",
    "Passport",
    "PitchSatisfactionScore",
    "OwnCar",
    "NumberOfChildrenVisiting",
    "MonthlyIncome",
]

categorical_features = [
    "TypeofContact",
    "Occupation",
    "Gender",
    "ProductPitched",
    "MaritalStatus",
    "Designation",
]

numeric_transformer = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
)

categorical_transformer = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore"),
)

preprocessor = make_column_transformer(
    (numeric_transformer, numeric_features),
    (categorical_transformer, categorical_features),
)

scale_pos_weight = ytrain.value_counts()[0] / ytrain.value_counts()[1]
xgb_model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
)

model_pipeline = make_pipeline(preprocessor, xgb_model)

param_grid = {
    "xgbclassifier__n_estimators": [100, 150],
    "xgbclassifier__max_depth": [3, 5],
    "xgbclassifier__learning_rate": [0.05, 0.1],
    "xgbclassifier__subsample": [0.8, 1.0],
    "xgbclassifier__colsample_bytree": [0.8, 1.0],
    "xgbclassifier__reg_lambda": [1, 5],
}

with mlflow.start_run():
    grid_search = GridSearchCV(
        model_pipeline,
        param_grid,
        cv=3,
        n_jobs=-1,
        scoring="f1",
    )
    grid_search.fit(Xtrain, ytrain)

    for i, params in enumerate(grid_search.cv_results_["params"]):
        with mlflow.start_run(nested=True):
            mlflow.log_params(params)
            mlflow.log_metric("mean_cv_f1", grid_search.cv_results_["mean_test_score"][i])
            mlflow.log_metric("std_cv_f1", grid_search.cv_results_["std_test_score"][i])

    best_model = grid_search.best_estimator_
    mlflow.log_params(grid_search.best_params_)
    mlflow.log_metric("best_cv_f1", grid_search.best_score_)

    threshold = 0.45
    train_proba = best_model.predict_proba(Xtrain)[:, 1]
    test_proba = best_model.predict_proba(Xtest)[:, 1]
    y_pred_train = (train_proba >= threshold).astype(int)
    y_pred_test = (test_proba >= threshold).astype(int)

    train_report = classification_report(ytrain, y_pred_train, output_dict=True)
    test_report = classification_report(ytest, y_pred_test, output_dict=True)

    tn, fp, fn, tp = confusion_matrix(ytest, y_pred_test).ravel()
    mlflow.log_metrics({
        "train_accuracy": train_report["accuracy"],
        "train_precision": train_report["1"]["precision"],
        "train_recall": train_report["1"]["recall"],
        "train_f1": train_report["1"]["f1-score"],
        "test_accuracy": test_report["accuracy"],
        "test_precision": test_report["1"]["precision"],
        "test_recall": test_report["1"]["recall"],
        "test_f1": test_report["1"]["f1-score"],
        "test_roc_auc": roc_auc_score(ytest, test_proba),
        "test_true_negative": tn,
        "test_false_positive": fp,
        "test_false_negative": fn,
        "test_true_positive": tp,
    })

    joblib.dump(best_model, MODEL_FILENAME)
    mlflow.log_artifact(MODEL_FILENAME, artifact_path="model")

    try:
        api.repo_info(repo_id=MODEL_REPO_ID, repo_type="model")
        print(f"Model repo '{MODEL_REPO_ID}' already exists. Using it.")
    except RepositoryNotFoundError:
        create_repo(repo_id=MODEL_REPO_ID, repo_type="model", private=False, token=hf_token)
        print(f"Model repo '{MODEL_REPO_ID}' created.")

    api.upload_file(
        path_or_fileobj=MODEL_FILENAME,
        path_in_repo=MODEL_FILENAME,
        repo_id=MODEL_REPO_ID,
        repo_type="model",
    )

print(f"Model saved and uploaded as {MODEL_FILENAME}.")


Writing tourism_project/model_building/train.py


# Deployment

## Dockerfile

In [13]:
os.makedirs("tourism_project/deployment", exist_ok=True)

In [14]:
%%writefile tourism_project/deployment/Dockerfile
# Use a minimal base image with Python 3.9 installed
FROM python:3.9

# Set the working directory inside the container to /app
WORKDIR /app

# Copy all files from the current directory on the host to the container's /app directory
COPY . .

# Install Python dependencies listed in requirements.txt
RUN pip3 install -r requirements.txt

RUN useradd -m -u 1000 user
USER user
ENV HOME=/home/user \
	PATH=/home/user/.local/bin:$PATH

WORKDIR $HOME/app

COPY --chown=user . $HOME/app

# Define the command to run the Streamlit app on port "8501" and make it accessible externally
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0", "--server.enableXsrfProtection=false"]

Writing tourism_project/deployment/Dockerfile


## Streamlit App

Please ensure that the web app script is named `app.py`.

In [15]:
%%writefile tourism_project/deployment/app.py
import joblib
import pandas as pd
import streamlit as st
from huggingface_hub import hf_hub_download

HF_USERNAME = "karthikei94"
MODEL_REPO_ID = f"{HF_USERNAME}/tourism-purchase-model"
MODEL_FILENAME = "best_tourism_purchase_model_v1.joblib"

@st.cache_resource
def load_model():
    model_path = hf_hub_download(repo_id=MODEL_REPO_ID, filename=MODEL_FILENAME)
    return joblib.load(model_path)

model = load_model()

st.title("Wellness Tourism Package Purchase Prediction")
st.write("Enter customer and interaction details to estimate whether the customer is likely to purchase the Wellness Tourism Package.")

col1, col2 = st.columns(2)

with col1:
    age = st.number_input("Age", min_value=18, max_value=100, value=35)
    typeof_contact = st.selectbox("Type of Contact", ["Self Enquiry", "Company Invited"])
    city_tier = st.selectbox("City Tier", [1, 2, 3])
    duration_of_pitch = st.number_input("Duration of Pitch", min_value=0.0, max_value=120.0, value=15.0, step=1.0)
    occupation = st.selectbox("Occupation", ["Salaried", "Small Business", "Large Business", "Free Lancer"])
    gender = st.selectbox("Gender", ["Female", "Male"])
    number_of_person_visiting = st.number_input("Number of Persons Visiting", min_value=1, max_value=10, value=3)
    number_of_followups = st.number_input("Number of Followups", min_value=0, max_value=10, value=3)
    product_pitched = st.selectbox("Product Pitched", ["Basic", "Deluxe", "Standard", "Super Deluxe", "King"])

with col2:
    preferred_property_star = st.selectbox("Preferred Property Star", [3, 4, 5])
    marital_status = st.selectbox("Marital Status", ["Single", "Married", "Divorced", "Unmarried"])
    number_of_trips = st.number_input("Number of Trips", min_value=0, max_value=30, value=2)
    passport = st.selectbox("Passport", [0, 1], format_func=lambda x: "Yes" if x == 1 else "No")
    pitch_satisfaction_score = st.slider("Pitch Satisfaction Score", min_value=1, max_value=5, value=3)
    own_car = st.selectbox("Own Car", [0, 1], format_func=lambda x: "Yes" if x == 1 else "No")
    number_of_children_visiting = st.number_input("Number of Children Visiting", min_value=0, max_value=10, value=1)
    designation = st.selectbox("Designation", ["Executive", "Manager", "Senior Manager", "AVP", "VP"])
    monthly_income = st.number_input("Monthly Income", min_value=0.0, max_value=200000.0, value=25000.0, step=1000.0)

input_data = pd.DataFrame([{
    "Age": age,
    "TypeofContact": typeof_contact,
    "CityTier": city_tier,
    "DurationOfPitch": duration_of_pitch,
    "Occupation": occupation,
    "Gender": gender,
    "NumberOfPersonVisiting": number_of_person_visiting,
    "NumberOfFollowups": number_of_followups,
    "ProductPitched": product_pitched,
    "PreferredPropertyStar": preferred_property_star,
    "MaritalStatus": marital_status,
    "NumberOfTrips": number_of_trips,
    "Passport": passport,
    "PitchSatisfactionScore": pitch_satisfaction_score,
    "OwnCar": own_car,
    "NumberOfChildrenVisiting": number_of_children_visiting,
    "Designation": designation,
    "MonthlyIncome": monthly_income,
}])

if st.button("Predict Purchase"):
    probability = model.predict_proba(input_data)[0, 1]
    prediction = int(probability >= 0.45)
    st.subheader("Prediction Result")
    if prediction == 1:
        st.success(f"Likely to purchase. Estimated probability: {probability:.1%}")
    else:
        st.info(f"Not likely to purchase. Estimated probability: {probability:.1%}")


Writing tourism_project/deployment/app.py


## Dependency Handling

Please ensure that the dependency handling file is named `requirements.txt`.

In [16]:
%%writefile tourism_project/deployment/requirements.txt
pandas==2.2.2
huggingface_hub==0.32.6
streamlit==1.43.2
joblib==1.5.1
scikit-learn==1.6.0
xgboost==2.1.4
mlflow==3.0.1


Writing tourism_project/deployment/requirements.txt


# Hosting

In [17]:
%%writefile tourism_project/hosting.py
from huggingface_hub import HfApi, create_repo
from huggingface_hub.utils import RepositoryNotFoundError
import os
from dotenv import load_dotenv

HF_USERNAME = "karthikei94"
SPACE_REPO_NAME = "wellness-tourism-purchase-prediction"
repo_id = f"{HF_USERNAME}/{SPACE_REPO_NAME}"

load_dotenv()

hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    raise ValueError("HF_TOKEN is missing. Add it to your .env file for local runs or GitHub secrets for Actions.")

api = HfApi(token=hf_token)

try:
    api.repo_info(repo_id=repo_id, repo_type="space")
    print(f"Space '{repo_id}' already exists. Using it.")
except RepositoryNotFoundError:
    create_repo(
        repo_id=repo_id,
        repo_type="space",
        space_sdk="streamlit",
        private=False,
        token=hf_token,
    )
    print(f"Space '{repo_id}' created.")

api.upload_folder(
    folder_path="tourism_project/deployment",
    repo_id=repo_id,
    repo_type="space",
    path_in_repo="",
)
print("Deployment files uploaded successfully.")


Writing tourism_project/hosting.py


# MLOps Pipeline with Github Actions Workflow

**Note:**

1. Before running the file below, make sure to add the HF_TOKEN to your GitHub secrets to enable authentication between GitHub and Hugging Face.
2. The below code is for a sample YAML file that can be updated as required to meet the requirements of this project.

## Complete GitHub Actions YAML Configuration

The workflow below automates dataset registration, data preparation, model training with MLflow tracking, model upload, and Streamlit app hosting on Hugging Face Spaces.


```
name: Tourism Project Pipeline

on:
  push:
    branches:
      - main

jobs:
  register-dataset:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: pip install -r tourism_project/requirements.txt
      - name: Copy Dataset into Project Folder
        run: |
          mkdir -p tourism_project/data
          cp tourism.csv tourism_project/data/tourism.csv
      - name: Upload Dataset to Hugging Face Hub
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python tourism_project/model_building/data_register.py

  data-prep:
    needs: register-dataset
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: pip install -r tourism_project/requirements.txt
      - name: Run Data Preparation
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python tourism_project/model_building/prep.py

  model-training:
    needs: data-prep
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: pip install -r tourism_project/requirements.txt
      - name: Start MLflow Server
        run: |
          nohup mlflow ui --host 0.0.0.0 --port 5000 &
          sleep 5
      - name: Model Building
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python tourism_project/model_building/train.py

  deploy-hosting:
    needs: [register-dataset, data-prep, model-training]
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: pip install -r tourism_project/requirements.txt
      - name: Push Files to Hugging Face Space
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python tourism_project/hosting.py
```


**Note:** To use this YAML file for our use case, we need to

1. Go to the GitHub repository for the project
2. Create a folder named ***.github/workflows/***
3. In the above folder, create a file named ***pipeline.yml***
4. Copy and paste the above content for the YAML file into the ***pipeline.yml*** file

## Requirements file for the Github Actions Workflow

In [ ]:
%%writefile tourism_project/requirements.txt
huggingface_hub==0.32.6
datasets==3.6.0
pandas==2.2.2
scikit-learn==1.6.0
xgboost==2.1.4
mlflow==3.0.1
joblib==1.5.1
python-dotenv==1.1.1


### Steps to Setup GitHub Actions

1. Create a GitHub repository for this project.
2. Add `HF_TOKEN` under repository `Settings > Secrets and variables > Actions`.
3. Create `.github/workflows/pipeline.yml` in the repository.
4. Paste the workflow YAML from the previous section into `pipeline.yml`.
5. Push `tourism.csv` and the generated `tourism_project` folder to the repository.

### How the Automated Workflow Works

The pipeline runs in four stages. First, it registers the raw dataset on Hugging Face. Second, it creates train/test datasets and uploads them. Third, it trains and logs an XGBoost model with MLflow. Finally, it uploads the Streamlit deployment files to Hugging Face Spaces.

### Monitoring Your Pipeline

After pushing code, open the repository's `Actions` tab to monitor each job. If a job fails, inspect the failed step logs, confirm `HF_TOKEN` is available, and verify that all placeholder repository names have been replaced with your Hugging Face username.


## Github Authentication and Push Files

* Before moving forward, we need to generate a secret token to push files directly from Colab to the GitHub repository.
* Please follow the below instructions to create the GitHub token:
    - Open your GitHub profile.
    - Click on ***Settings***.
    - Go to ***Developer Settings***.
    - Expand the ***Personal access tokens*** section and select ***Tokens (classic)***.
    - Click ***Generate new token***, then choose ***Generate new token (classic)***.
    - Add a note and select all required scopes.
    - Click ***Generate token***.
    - Copy the generated token and store it safely in a notepad.

In [ ]:
# Install Git
!apt-get install git

# Set your Git identity (replace with your details)
!git config --global user.email "<-------GitHub Email Address------->"
!git config --global user.name "<--------GitHub UserName--------->"

# Clone your GitHub repository
!git clone https://github.com/<--------GitHub UserName--------->/<--------GitHub Reponame--------->.git

# Move your folder to the repository directory
!mv /content/tourism_project/ /content/<--------GitHub Reponame--------->

In [ ]:
# Change directory to the cloned repository
%cd <--------GitHub Reponame--------->/

# Add the new folder to Git
!git add .

# Commit the changes
!git commit -m "first commit"

# Push to GitHub (you'll need your GitHub credentials; use a personal access token if 2FA enabled)
!git push https://<--------GitHub UserName--------->:<--------GitHub Token--------->@github.com/<--------GitHub UserName--------->/<--------GitHub Reponame--------->.git

# Output Evaluation

## GIT Folder Struture

Include a screenshot showing the generated `tourism_project` folder, the model building scripts, deployment files, requirements files, and `.github/workflows/pipeline.yml`.

## Github actions

Include a screenshot of the successful GitHub Actions workflow run showing all jobs completed.


- GitHub (link to repository, screenshot of folder structure and executed workflow)

In [ ]:
# Add your GitHub repository link, folder structure screenshot, and executed workflow screenshot here.


- Streamlit on Hugging Face (link to HF space, screenshot of Streamlit app)

In [ ]:
# Add your Hugging Face Space link and Streamlit app screenshot here.


<font size=6 color="navyblue">Power Ahead!</font>
___